In [1]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv('english.csv')
chars74k_alphabet = set(df['label'].astype(str).unique())
print(sorted(chars74k_alphabet))
print(len(chars74k_alphabet))

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
62


In [6]:
text = """Dear diary, today I finally sat down to practice my handwriting again. The pen felt heavy in my hand at first, but after a few lines it began to flow more naturally across the page. I wrote about the morning walk I took through the old part of town, past the bakery with the cracked window and the small park where the children were feeding the pigeons. The letters looped and slanted the way they always do when I write quickly, and I noticed how my capital letters tend to lean forward while my lowercase letters stay upright and small.

Later in the afternoon I copied out a few lines from a book I had been reading, just to keep my hand steady and my pen moving. Copying text like this is a strange kind of meditation. You stop thinking about the meaning of the words and start thinking only about the shape of each letter, the curve of an a, the loop of an l, the small dot that must be placed above every i and every j. It is easy to forget how many decisions go into forming a single word by hand.

In the evening I wrote a letter to an old friend. I told her about the garden, about the tomatoes finally turning red after weeks of green, and about the neighbor's cat that keeps sleeping on our porch steps as if it owns the place. I signed the letter the way I always do, with a small flourish under my name, a habit I picked up from my grandmother many years ago. She used to say that handwriting was a kind of portrait, that no two people ever write the same word in exactly the same way, and that a page of handwriting could tell you almost as much about a person as their face could.

I think about that often now, especially as more of the world moves toward typing instead of writing. There is something patient about handwriting, something that cannot be rushed without becoming messy. Each letter takes a small amount of time and care, and the result carries the trace of that time. A typed letter looks the same no matter who typed it. A handwritten letter always looks like it came from somewhere, from someone, from a particular afternoon with a particular pen.

Tomorrow I plan to practice cursive again, focusing on connecting my letters more smoothly, especially between the vowels and the consonants that tend to trip me up, like q and u, or the tricky transition from a lowercase g into the next letter. I also want to try writing with my left hand for fun, just to see how strange and unfamiliar it feels compared to my usual right hand. Maybe I will fill an entire page with numbers instead of letters, practicing zero through nine over and over until they look consistent and clean.

The old notebooks in the drawer are full of these little practice pages. Rows of the alphabet, capital and lowercase, over and over. Rows of digits. Short sentences repeated until the ink runs low. It is strange to flip through years of my own handwriting and see how much it has changed, how the letters used to be rounder and slower, and how now they are quicker, sharper, more slanted, shaped by thousands of hours of writing grocery lists, birthday cards, and quiet notes to myself.

Someday I would like to teach a machine to write the way I do, not just to recognize the letters but to imagine new ones, new words, new sentences that still look and feel like they came from the same hand. It seems like a strange goal, but there is something appealing about it, the idea that a pattern learned from enough examples could produce something new that still carries a familiar style."""

print(f"Corpus length: {len(text)} characters")
print(text[:300])

Corpus length: 3496 characters
Dear diary, today I finally sat down to practice my handwriting again. The pen felt heavy in my hand at first, but after a few lines it began to flow more naturally across the page. I wrote about the morning walk I took through the old part of town, past the bakery with the cracked window and the sm


In [7]:
chars = sorted(set(text) | chars74k_alphabet)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(chars)

Vocabulary size: 67
['\n', ' ', "'", ',', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [8]:
data_idx = [char_to_idx[ch] for ch in text]
print(f"Total training characters: {len(data_idx)}")
print(data_idx[:20])

Total training characters: 3496
[18, 45, 41, 58, 1, 44, 49, 41, 58, 65, 3, 1, 60, 55, 44, 41, 65, 1, 23, 1]


In [9]:
np.random.seed(42)

hidden_size = 128
seq_length = 25
learning_rate = 0.1

Wxh = np.random.randn(hidden_size, vocab_size) * 0.01   # input -> hidden
Whh = np.random.randn(hidden_size, hidden_size) * 0.01  # hidden -> hidden
Why = np.random.randn(vocab_size, hidden_size) * 0.01   # hidden -> output
bh = np.zeros((hidden_size, 1))                          # hidden bias
by = np.zeros((vocab_size, 1))                            # output bias

print("Model initialized.")
print("Wxh shape:", Wxh.shape)
print("Whh shape:", Whh.shape)
print("Why shape:", Why.shape)

Model initialized.
Wxh shape: (128, 67)
Whh shape: (128, 128)
Why shape: (67, 128)


In [11]:
def loss_and_grads(inputs, targets, h_prev):
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(h_prev)
    loss = 0

    for t in range(len(inputs)):
        xs[t] = np.zeros((vocab_size, 1))
        xs[t][inputs[t]] = 1
        hs[t] = np.tanh(Wxh @ xs[t] + Whh @ hs[t-1] + bh)
        ys[t] = Why @ hs[t] + by
        exp_y = np.exp(ys[t] - np.max(ys[t]))
        ps[t] = exp_y / np.sum(exp_y)
        loss += -np.log(ps[t][targets[t], 0] + 1e-12)

    dWxh = np.zeros_like(Wxh)
    dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why)
    dbh = np.zeros_like(bh)
    dby = np.zeros_like(by)
    dh_next = np.zeros_like(hs[0])

    for t in reversed(range(len(inputs))):
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1
        dWhy += dy @ hs[t].T
        dby += dy
        dh = Why.T @ dy + dh_next
        dh_raw = (1 - hs[t] ** 2) * dh
        dbh += dh_raw
        dWxh += dh_raw @ xs[t].T
        dWhh += dh_raw @ hs[t-1].T
        dh_next = Whh.T @ dh_raw

    for grad in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(grad, -5, 5, out=grad)

    return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

print("loss_and_grads function defined.")

loss_and_grads function defined.


In [12]:
def sample(h, seed_idx, n, temperature=0.8):
    x = np.zeros((vocab_size, 1))
    x[seed_idx] = 1
    indices = []

    for t in range(n):
        h = np.tanh(Wxh @ x + Whh @ h + bh)
        y = Why @ h + by
        y = y / temperature
        exp_y = np.exp(y - np.max(y))
        p = (exp_y / np.sum(exp_y)).ravel()
        idx = np.random.choice(range(vocab_size), p=p)
        x = np.zeros((vocab_size, 1))
        x[idx] = 1
        indices.append(idx)

    return ''.join(idx_to_char[i] for i in indices)

print("sample function defined.")

sample function defined.


In [13]:
mWxh = np.zeros_like(Wxh)
mWhh = np.zeros_like(Whh)
mWhy = np.zeros_like(Why)
mbh = np.zeros_like(bh)
mby = np.zeros_like(by)

print("Adagrad memory initialized.")

Adagrad memory initialized.


In [14]:
n_iters = 20000
print_every = 1000

p = 0
h_prev = np.zeros((hidden_size, 1))
smooth_loss = -np.log(1.0 / vocab_size) * seq_length

for it in range(1, n_iters + 1):
    if p + seq_length + 1 >= len(data_idx) or it == 1:
        h_prev = np.zeros((hidden_size, 1))
        p = 0

    inputs = data_idx[p : p + seq_length]
    targets = data_idx[p + 1 : p + seq_length + 1]

    loss, dWxh, dWhh, dWhy, dbh, dby, h_prev = loss_and_grads(inputs, targets, h_prev)

    for param, dparam, mem in zip(
        [Wxh, Whh, Why, bh, by],
        [dWxh, dWhh, dWhy, dbh, dby],
        [mWxh, mWhh, mWhy, mbh, mby]
    ):
        mem += dparam * dparam
        param += -learning_rate * dparam / np.sqrt(mem + 1e-8)

    smooth_loss = smooth_loss * 0.999 + loss * 0.001
    p += seq_length

    if it % print_every == 0 or it == 1:
        print(f"iter {it:6d} | smooth loss {smooth_loss:.4f}")
        sample_text = sample(h_prev, inputs[0], 150, temperature=0.8)
        print("  sample:", repr(sample_text[:120]))

iter      1 | smooth loss 105.1173
  sample: "FvGYgGHPn1YbVvWYXf3s'aX8'3Cqp,Dr9zBgL3AbGX7NRL7Ocxj5C.3A789D2Ce95.Qn,SaWqp,Hcwk5CiPTX1ppOYa62 uG0P.0QRWaaMaENKFA.RTFyTrU"
iter   1000 | smooth loss 87.8773
  sample: 'aseinu ms  ywolcand me, te nanl un lootiL r nraliatheatc tomeutdea dVr teadaleeepno weeeny iacl thafaus aicirer mn ine n'
iter   2000 | smooth loss 74.1039
  sample: ' m he s wtiull thaw, pan tusr f panr aathas osen an the the ond n I loowd s peasitt. I an nanatinr thd ethan atheralI m '
iter   3000 | smooth loss 65.9014
  sample: 'tertharg ls cetto gose I ttoter umsif ghe m wes terroe the bhe sioudow y thase boong lrsoy I ans the anson co sindesof y'
iter   4000 | smooth loss 60.7991
  sample: 'peshe fow Took lyt whe ther y I ol ofpeithe le tothed tars hotl Qfeetht ace racheso the sal ons thand po. Tanl of the po'
iter   5000 | smooth loss 57.3909
  sample: 'ke ling the lhean torof of toulgertCis martos rrinjutl thang wat ghe wouckre ove staed ane mrand soowe int ard I no ten

In [15]:
h = np.zeros((hidden_size, 1))
seed = char_to_idx[text[0]]
generated = sample(h, seed, 600, temperature=0.7)

print("=== Final generated text (temperature=0.7) ===\n")
print(generated)

=== Final generated text (temperature=0.7) ===

ear diarere no hond the nind tine now end hok cartin on bowed ars tortine took inke l and hot beI til imatred cof sand of serte sead paet mom abom of hat of writine somy agrlcougring mrate pand and now the lors qu the cand of thang intided  fme and uped bouel thing atd ploun, the shard fanl mribevea go smare lorting thand a and hape libust ayped. I wit lersooker and sy theo, and ho thars the well of the doacry plansters a warind Is feace hourstontterhing all up, the lernors bersing, pace poop mure of cars the ill uls the lest sead of ghene hike aticf juld and hanten sowinnte sreens ofrliser an


In [16]:
for temp in [0.5, 0.8, 1.1]:
    h = np.zeros((hidden_size, 1))
    seed = char_to_idx[text[0]]
    print(f"\n=== temperature={temp} ===")
    print(sample(h, seed, 300, temperature=temp))


=== temperature=0.5 ===
ear tarow the rort lersing do tors the pand and now feespsackecers alth I witing. The ting mersing dernooke q and hort fracou the caprting wricing afrcer and how the no s fies thace teanger and houthe sharthang allo, just and plas lercalrow and I com letters and it obe teret and no bow mers to old a

=== temperature=0.8 ===
ear diarower, an of che I wiys dite none. Moy thace a funt legterater bow slinf ffougoter ine therand tor and the nors ald lers a pat bow the nome pring morpary smiset ghe garse y wherace where ans lop the, le mowitins ghe send throg andw nerspand lange ane on oo saclers quoquth thay and in smalpeds

=== temperature=1.1 ===
ear dardicrougomr sallo, ke ds the wa d. Thirs beal, thachcousd,, lome ingh s anped.


Lr moutter ase cme sera firs theo, I of of the ocested. Sesler are chat lesl, asl.


ZTheus the sepind nom routif thousanke si han, tele over, and, of like 
he alitize t, chatkedwind pary at toe and sorver soweed 
